In [ ]:
!pip install pandas

In [ ]:
import urllib.request
import pandas as pd

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/ops4life/mlops-get-started/main/datasets/HR-Employee-Attrition.csv",
    "HR-Employee-Attrition.csv"
)
# Run prior pipeline steps (ingest + validate) to produce cleaned.csv
df_raw = pd.read_csv("HR-Employee-Attrition.csv")
df_raw.to_csv("cleaned.csv", index=False)
print("Setup complete: cleaned.csv ready.")

In [ ]:
import pandas as pd

def feature_data(df):
    df_fe = df.copy()

    # Encoding
    # ------------------------------------
    # 1. Target column
    df_fe['Attrition'] = df_fe['Attrition'].map({'Yes': 1, 'No': 0}).astype('int')

    # 2. Binary Encoding
    df_fe['OverTime'] = df_fe['OverTime'].map({'Yes': 1, 'No': 0}).astype('Int64')
    df_fe['Gender'] = df_fe['Gender'].map({'Male': 1, 'Female': 0}).astype('Int64')

    # 3. Ordinal encoding
    df_fe['BusinessTravel'] = df_fe['BusinessTravel'].map({'Non-Travel': 0, 'Travel_Rarely': 1, 'Travel_Frequently': 2}).astype('Int64')
    df_fe['MaritalStatus'] = df_fe['MaritalStatus'].map({'Single': 0, 'Married': 1, 'Divorced': 2}).astype('Int64')

    # WorkLifeBalance, JobSatisfaction, EnvironmentSatisfaction, RelationshipSatisfaction,
    # JobLevel, PerformanceRating, JobInvolvement are already numeric (1–N), no encoding needed.

    # Aggregate satisfaction
    df_fe['OverallSatisfaction'] = (
        (df_fe['JobSatisfaction'] + df_fe['EnvironmentSatisfaction'] + df_fe['RelationshipSatisfaction'] + df_fe['WorkLifeBalance']) / 4
    ).round().astype('Int64')
    df_fe = df_fe.drop(columns=['JobSatisfaction', 'EnvironmentSatisfaction', 'RelationshipSatisfaction', 'WorkLifeBalance'])

    # Annual income binning
    df_fe['AnnualIncome'] = pd.cut(
        df_fe['MonthlyIncome'] * 12,
        bins=[0, 30000, 60000, 90000, 120000, float('inf')],
        labels=[0, 1, 2, 3, 4],
        include_lowest=True
    ).astype('Int64')
    df_fe = df_fe.drop(columns=['MonthlyIncome'])

    # Age binning
    df_fe['AgeGroup'] = pd.cut(
        df_fe['Age'],
        bins=[17, 25, 35, 45, 60, 65],
        labels=[1, 2, 3, 4, 5]
    ).astype('Int64')
    df_fe = df_fe.drop(columns=['Age'])

    # Derived features — HR dataset already in years, no /12 needed
    # 1. Role Stagnation: high -> same role entire tenure (possible stagnation)
    df_fe['RoleStagnationRatio'] = (df_fe['YearsInCurrentRole'] / (df_fe['YearsAtCompany'] + 1)).round(3)

    # 2. Tenure Gap: high gap -> worked elsewhere before; low gap -> grew with this company
    df_fe['TenureGap'] = df_fe['TotalWorkingYears'] - df_fe['YearsAtCompany']

    # 3. Early company risk: most attrition happens in first 2 years
    df_fe['EarlyCompanyTenureRisk'] = (df_fe['YearsAtCompany'] <= 2).astype('Int64')

    # 4. Long Term Stagnation
    df_fe['LongTenureLowRoleRisk'] = ((df_fe['TotalWorkingYears'] > 5) & (df_fe['JobLevel'] <= 2)).astype('Int64')

    # Drop ID/constant columns and high-cardinality categoricals
    drop_cols = [c for c in [
        'EmployeeNumber', 'EmployeeCount', 'StandardHours', 'Over18',
        'JobRole', 'EducationField', 'Department'
    ] if c in df_fe.columns]
    df_fe = df_fe.drop(columns=drop_cols)

    print(df_fe.tail(10))

    df_fe.to_csv("featured.csv", index=False)

    return df_fe


df = pd.read_csv("cleaned.csv")
feature_data(df)